In [81]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, regexp_replace
from pyspark.sql.types import ArrayType, StructType, StructField, IntegerType, StringType
from pyspark.sql import functions as F
from pyspark.sql.functions import col  # the count is every user's rating counting
from pyspark.sql.functions import count
from scipy.sparse import csr_matrix
import numpy as np
from sklearn.neighbors import NearestNeighbors

# stop any existing Spark session, if Spark is already running, creating a new session might fail ***
try:
    spark.stop()
except Exception:
    pass

# create session with adjusted memory settings based on your cluster
# .config("spark.local.dir", r"E:\Apache Spark\spark-temp"): change the spark local dir, as the c disk memory is not enough, may cause Py4JJavaError exception ***
spark = SparkSession.builder.appName("MovieRecommender_SQL_Movielens") \
    .config("spark.executor.memory", "32g") \
    .config("spark.driver.memory", "16g") \
    .config("spark.local.dir", r"E:\Apache Spark\spark-temp") \
    .config("spark.jars", r"D:\MySQL\MySQL ConnectorJ\mysql-connector-j-8.4.0\mysql-connector-j-8.4.0\mysql-connector-j-8.4.0.jar") \
    .getOrCreate()

In [82]:
spark

### Read Movie and Rating Data from MySQL

In [83]:
# Read data from MySQL
movies = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:mysql://localhost:3307/db_movie_recommender_sys?useSSL=false&serverTimezone=UTC") \
    .option("dbtable", "tb_movie") \
    .option("user", "root") \
    .option("password", "137162") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

In [84]:
movies.show()

+--------+--------------------+--------------------+
|movie_id|               title|              genres|
+--------+--------------------+--------------------+
|       1|    Toy Story (1995)|Adventure|Animati...|
|       2|      Jumanji (1995)|Adventure|Childre...|
|       3|Grumpier Old Men ...|      Comedy|Romance|
|       4|Waiting to Exhale...|Comedy|Drama|Romance|
|       5|Father of the Bri...|              Comedy|
|       6|         Heat (1995)|Action|Crime|Thri...|
|       7|      Sabrina (1995)|      Comedy|Romance|
|       8| Tom and Huck (1995)|  Adventure|Children|
|       9| Sudden Death (1995)|              Action|
|      10|    GoldenEye (1995)|Action|Adventure|...|
|      11|American Presiden...|Comedy|Drama|Romance|
|      12|Dracula: Dead and...|       Comedy|Horror|
|      13|        Balto (1995)|Adventure|Animati...|
|      14|        Nixon (1995)|               Drama|
|      15|Cutthroat Island ...|Action|Adventure|...|
|      16|       Casino (1995)|         Crime|

In [85]:
movies.select("title", "genres").dropDuplicates(['title']).count()

27262

In [86]:
ratings = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:mysql://localhost:3307/db_movie_recommender_sys?useSSL=false&serverTimezone=UTC") \
    .option("dbtable", "tb_rating") \
    .option("user", "root") \
    .option("password", "137162") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

In [87]:
rating_count = ratings.groupBy("user_id") \
       .count() \
       .orderBy(col("count"), ascending=False)

In [88]:
rating_count = rating_count.filter(rating_count['count'] > 200)

In [89]:
user_ids = rating_count.select("user_id")

user_id_list = [row['user_id'] for row in user_ids.collect()]  # get the user id list

In [90]:
# filters the ratings DataFrame to include only rows with user IDs that are in y—that is, 
# only ratings by users with more than 200 ratings.
ratings = ratings.filter(ratings.user_id.isin(user_id_list))

### Integrate Moive and Rating Data

In [91]:
# join the ratings with the movies
ratings_with_movies = ratings.join(movies, on="movie_id", how="inner")

ratings_with_movies = ratings_with_movies.dropDuplicates(['movie_id', 'user_id']) # this is quite important, each user can rate the same movie multiple times

In [92]:
# The agg() function in this code is used to apply one or more aggregate functions to a DataFrame after grouping it by a specific column (in this case, "title").

rating_count_every_title = ratings_with_movies.groupBy('title').agg(count('rating').alias('rating_count'))

In [93]:
ratings_with_movies = ratings_with_movies.join(rating_count_every_title, on='title', how='inner')

ratings_with_movies = ratings_with_movies.filter(ratings_with_movies['rating_count'] >= 50)

In [94]:
ratings_with_movies.show()

+----------------+--------+-------+------+-------------------+-------+--------------------+------------+
|           title|movie_id|user_id|rating|          timestamp|     id|              genres|rating_count|
+----------------+--------+-------+------+-------------------+-------+--------------------+------------+
|Toy Story (1995)|       1|    741|   5.0|2007-10-16 21:51:03| 419582|Adventure|Animati...|          58|
|Toy Story (1995)|       1|    775|   4.5|2005-04-18 04:03:46| 480113|Adventure|Animati...|          58|
|Toy Story (1995)|       1|   1849|   4.5|2006-01-31 00:33:19| 112683|Adventure|Animati...|          58|
|Toy Story (1995)|       1|   3858|   4.0|2010-02-22 00:11:57| 888349|Adventure|Animati...|          58|
|Toy Story (1995)|       1|   4358|   5.0|2000-12-06 02:33:30|1318707|Adventure|Animati...|          58|
|Toy Story (1995)|       1|   5843|   5.0|2005-03-17 20:10:58|1688566|Adventure|Animati...|          58|
|Toy Story (1995)|       1|   6099|   4.0|2005-03-23 11

### Movie Recommender Enigine By Genres

In [ ]:
# Add these global variables
genre_sparse = None
genre_knn_model = None
genre_movie_titles = []
genre_title_to_index = {}

In [ ]:
# process genres for content-based filtering
# extract unique movies with genres from the filtered ratings_with_movies
unique_movies_with_genres = ratings_with_movies.select("title", "genres").dropDuplicates(['title'])

In [97]:
unique_movies_with_genres.show() 

+--------------------+--------------------+
|               title|              genres|
+--------------------+--------------------+
|2001: A Space Ody...|Adventure|Drama|S...|
|28 Days Later (2002)|Action|Horror|Sci-Fi|
|A.I. Artificial I...|Adventure|Drama|S...|
|  About a Boy (2002)|Comedy|Drama|Romance|
|    Airplane! (1980)|              Comedy|
|      Aladdin (1992)|Adventure|Animati...|
|       Aliens (1986)|Action|Adventure|...|
|Alien³ (a.k.a. Al...|Action|Horror|Sci...|
|Almost Famous (2000)|               Drama|
|      Amadeus (1984)|               Drama|
|American Beauty (...|        Comedy|Drama|
|American Graffiti...|        Comedy|Drama|
|American Presiden...|Comedy|Drama|Romance|
|   Annie Hall (1977)|      Comedy|Romance|
|         Antz (1998)|Adventure|Animati...|
|Apocalypse Now (1...|    Action|Drama|War|
|    Apollo 13 (1995)|Adventure|Drama|IMAX|
|   Armageddon (1998)|Action|Romance|Sc...|
|As Good as It Get...|Comedy|Drama|Romance|
|Austin Powers: In...|Action|Adv

In [98]:
unique_movies_with_genres.count()

213

In [99]:
# split genres into array and explode to individual rows

movies_genres = unique_movies_with_genres.withColumn("genre", F.explode(F.split(F.col("genres"), r"\|")))

In [100]:
movies_genres.show()

+--------------------+--------------------+---------+
|               title|              genres|    genre|
+--------------------+--------------------+---------+
|2001: A Space Ody...|Adventure|Drama|S...|Adventure|
|2001: A Space Ody...|Adventure|Drama|S...|    Drama|
|2001: A Space Ody...|Adventure|Drama|S...|   Sci-Fi|
|28 Days Later (2002)|Action|Horror|Sci-Fi|   Action|
|28 Days Later (2002)|Action|Horror|Sci-Fi|   Horror|
|28 Days Later (2002)|Action|Horror|Sci-Fi|   Sci-Fi|
|A.I. Artificial I...|Adventure|Drama|S...|Adventure|
|A.I. Artificial I...|Adventure|Drama|S...|    Drama|
|A.I. Artificial I...|Adventure|Drama|S...|   Sci-Fi|
|  About a Boy (2002)|Comedy|Drama|Romance|   Comedy|
|  About a Boy (2002)|Comedy|Drama|Romance|    Drama|
|  About a Boy (2002)|Comedy|Drama|Romance|  Romance|
|    Airplane! (1980)|              Comedy|   Comedy|
|      Aladdin (1992)|Adventure|Animati...|Adventure|
|      Aladdin (1992)|Adventure|Animati...|Animation|
|      Aladdin (1992)|Advent

In [101]:
# pivot to create binary genre columns
genre_pivot = movies_genres.groupBy("title").pivot("genre").agg(F.count("genre")).fillna(0)

In [102]:
genre_pivot.show()

+--------------------+------+---------+---------+--------+------+-----+-----------+-----+-------+---------+------+----+-------+-------+-------+------+--------+---+-------+
|               title|Action|Adventure|Animation|Children|Comedy|Crime|Documentary|Drama|Fantasy|Film-Noir|Horror|IMAX|Musical|Mystery|Romance|Sci-Fi|Thriller|War|Western|
+--------------------+------+---------+---------+--------+------+-----+-----------+-----+-------+---------+------+----+-------+-------+-------+------+--------+---+-------+
|2001: A Space Ody...|     0|        1|        0|       0|     0|    0|          0|    1|      0|        0|     0|   0|      0|      0|      0|     1|       0|  0|      0|
|28 Days Later (2002)|     1|        0|        0|       0|     0|    0|          0|    0|      0|        0|     1|   0|      0|      0|      0|     1|       0|  0|      0|
|A.I. Artificial I...|     0|        1|        0|       0|     0|    0|          0|    1|      0|        0|     0|   0|      0|      0|     

In [103]:
# convert to Pandas DataFrame for processing
genre_pivot_pd = genre_pivot.toPandas()


#### Code explains in Detail

```python
genre_columns = [col for col in genre_pivot_pd.columns if col != 'title']
```

`genre_pivot_pd.columns` is an attribute of the Pandas DataFrame `genre_pivot_pd`. It returns an `Index` object containing the names of all columns in the DataFrame.

```python
genre_pivot_pd.columns
```

```
Index(['title', 'Action', 'Adventure', 'Sci-Fi'], dtype='object')
```

```python
[col for col in genre_pivot_pd.columns if col != 'title']
```

- `for col in genre_pivot_pd.columns` iterates over all the column names in `genre_pivot_pd.columns`.
- `if col != 'title'` is a condition that filters out the `'title'` column. This means that only the columns that are not named `'title'` will be included in the resulting list.

After the list comprehension runs, the result is a list of column names, excluding `'title'`.
```

In [104]:
# get list of all genre columns (excludes the 'title' column)
genre_columns = [col for col in genre_pivot_pd.columns if col != 'title']

In [105]:
# convert genre indicators to a numerical matrix
# each row represents a movie, each column represents a genre
# values are 1 (has genre) or 0 (doesn't have genre) stored as 8-bit integers
genre_matrix = genre_pivot_pd[genre_columns].values.astype(np.int8)

# convert to sparse matrix format for efficient storage and computation
# CSR (Compressed Sparse Row) format is optimal for row-based operations
genre_sparse = csr_matrix(genre_matrix)

In [106]:
print(genre_sparse.toarray())

[[0 1 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 ...
 [1 1 0 ... 0 0 0]
 [1 1 0 ... 1 0 0]
 [0 0 0 ... 0 0 0]]


In [107]:
# create a list of movie titles preserving matrix order
# this maintains alignment between matrix rows and titles
genre_movie_titles = genre_pivot_pd['title'].tolist()

In [115]:
genre_movie_titles

['2001: A Space Odyssey (1968)',
 '28 Days Later (2002)',
 'A.I. Artificial Intelligence (2001)',
 'About a Boy (2002)',
 'Airplane! (1980)',
 'Aladdin (1992)',
 'Aliens (1986)',
 'Alien³ (a.k.a. Alien 3) (1992)',
 'Almost Famous (2000)',
 'Amadeus (1984)',
 'American Beauty (1999)',
 'American Graffiti (1973)',
 'American President, The (1995)',
 'Annie Hall (1977)',
 'Antz (1998)',
 'Apocalypse Now (1979)',
 'Apollo 13 (1995)',
 'Armageddon (1998)',
 'As Good as It Gets (1997)',
 'Austin Powers: International Man of Mystery (1997)',
 'Austin Powers: The Spy Who Shagged Me (1999)',
 'Awakenings (1990)',
 'Back to the Future (1985)',
 'Back to the Future Part II (1989)',
 'Batman & Robin (1997)',
 'Batman (1989)',
 'Batman Begins (2005)',
 'Batman Forever (1995)',
 'Beautiful Mind, A (2001)',
 'Being John Malkovich (1999)',
 'Best in Show (2000)',
 'Beverly Hills Cop (1984)',
 'Big (1988)',
 'Big Fish (2003)',
 'Big Lebowski, The (1998)',
 "Bill & Ted's Excellent Adventure (1989)",
 'B

In [ ]:
# create reverse mapping: title -> matrix row index
# enables quick lookup of a movie's position in the matrix
genre_title_to_index = {title: idx for idx, title in enumerate(genre_movie_titles)}

In [111]:
# train KNN model for genres using Jaccard similarity
genre_knn_model = NearestNeighbors(algorithm='brute')
genre_knn_model.fit(genre_sparse)

print("Genre-based recommender initialized!")

Genre-based recommender initialized!


In [ ]:
def recommend_by_genres(movie_title):
    global genre_title_to_index, genre_knn_model, genre_sparse, genre_movie_titles

    if not genre_title_to_index:
        return ["System not initialized"]

    idx = genre_title_to_index.get(movie_title)
    if idx is None:
        return ["Movie not found in genre database"]

    # find nearest neighbors
    distances, indices = genre_knn_model.kneighbors(genre_sparse[idx], n_neighbors=11)  # Get more to filter out

    # exclude the movie itself and return top 10
    recommendations = [genre_movie_titles[i] for i in indices[0] if i != idx][:10]
    return recommendations

In [114]:
print(recommend_by_genres("Stargate (1994)"))

['X-Men (2000)', 'Waterworld (1995)', 'Star Wars: Episode I - The Phantom Menace (1999)', 'Star Wars: Episode VI - Return of the Jedi (1983)', 'Star Wars: Episode IV - A New Hope (1977)', 'Superman (1978)', 'Aliens (1986)', 'Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)', 'Indiana Jones and the Last Crusade (1989)', 'Independence Day (a.k.a. ID4) (1996)']
